# Week 10 - Day 3 Lab
# AI Agent Frameworks with LangChain & LangGraph

---

## Learning Objectives

By the end of this lab, you will be able to:

- Build an AI Agent using LangChain.
- Register and use custom tools.
- Understand how LangChain simplifies the manual agent loop.
- Build a simple workflow using LangGraph.
- Add memory to an AI Agent.
- Extend the agent with your own custom tool.

Estimated Time: 2.5 Hours

# Step 1: Install Required Libraries

Run the following cell to install all required libraries.

In [1]:
!pip -q install -U langchain
!pip -q install -U langgraph
!pip -q install -U langchain-google-genai

# Step 2: Configure Gemini

We will use Google's Gemini model throughout this lab.

In [2]:
# load your key
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Create the Gemini LLM

Import the required class and initialize the model.

💡 Hint

The class name starts with

ChatGoogle...

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

### Test Your Model

If everything is configured correctly, the model should respond.

Expected Output

Hello! How can I help you today?

In [4]:
response = llm.invoke("Say Hello!")
print(response.content)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPxo9Ac4KodBZt3dI8T4Q/7lTTRKf209PCVgPT6xjDKfmTMD+23t+kGq6CWdcyyWGgbWSyctXzf5kMW7/jkBK77mJCdrIWSRxhRAGP3fr993hQ5bc7BJHr'}}]


# Creating Your First Tool

A tool is simply a Python function decorated with `@tool`.

Today we'll build a Weather Tool.

## Task

Complete the dictionary below.

💡 Hint

The dictionary key should be the city name.

The value should be the weather.

In [5]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Returns the weather of a city.
    """

    weather = {
        "Lahore": "Sunny, 34°C",
        "Karachi": "Humid, 31°C",
        "Islamabad": "Cloudy, 27°C",
        "Peshawar": "Hot, 36°C",
        "Quetta": "Cool, 22°C"
    }

    return weather.get(city, "Weather unavailable")

## Test Your Tool

In [6]:
print(get_weather.invoke("Lahore"))

Sunny, 34°C


## Challenge

Add TWO more cities.

Suggested cities:

- Peshawar
- Quetta

# Calculator Tool

Let's create another tool.

## Task

Complete the function.

💡 Hint

Python has a function that evaluates mathematical expressions.

In [7]:
@tool
def calculator(expression: str):
    """
    Evaluate a mathematical expression.
    """
    return eval(expression)

## Test

Try

25*18

Expected Output

450

In [8]:
calculator.invoke("25*18")

450

# Registering Tools

The AI Agent needs to know which tools it can use.

## Task

Complete the list below.

💡 Hint

Use the function names only.

In [9]:
tools = [
    get_weather,
    calculator
]

# Binding Tools to Gemini

Gemini must know which tools are available.

## Task

Complete the following line.

💡 Hint

The method starts with

bind...

In [10]:
llm_with_tools = llm.bind_tools(tools)

## Test Tool Calling

Ask Gemini about the weather.

Observe the response carefully.

Does Gemini answer directly?

Or

Does it request a tool?

In [11]:
response = llm_with_tools.invoke(

    "What's the weather in Lahore?"

)

response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Lahore"}'}, '__gemini_function_call_thought_signatures__': {'call_358399': 'El4KXAERTTIPVrtqx5vPR21NMCzXU/kpBPAxRoXjvMORrdna2Jj7AJC55zKRSWf6hrl0tfArWox3VJEcssqBYY0octSlk9vvqQwYI9Hse3ntjVb6vHCPHbySDVU7CG4/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06799-0b44-7963-abbc-be41009132ac-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': 'call_358399', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 17, 'total_tokens': 110, 'input_token_details': {'cache_read': 0}})

# Tool Registry

When Gemini requests a tool, Python needs to know which function to execute.

## Task

Complete the registry.

💡 Hint

The key is a string.

The value is the function.

In [12]:
tool_registry = {
    "get_weather": get_weather,
    "calculator": calculator
}

# Executing Tool Calls

Complete the missing lines.

💡 Hint

1. Find the tool.

2. Execute it.

3. Print the result.

In [13]:
for tool_call in response.tool_calls:

    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool = tool_registry[tool_name]

    result = tool.invoke(args)

    print(result)

Sunny, 34°C


# Creating a LangChain Agent

Earlier we manually:

- Parsed tool calls
- Executed tools
- Returned observations

LangChain automates all of that.

Complete the missing imports.

💡 Hint

The required classes are:

- AgentExecutor
- create_tool_calling_agent

In [14]:
!pip -q install -U langchain-classic

In [15]:
# TODO
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent

## Build the Agent

Fill in the blanks.

Don't worry if you don't remember every function—we discussed this in today's lecture.

💡 Hint

Use:

- llm
- tools
- prompt

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [17]:
agent = create_tool_calling_agent(llm, tools, prompt)

In [18]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## Test the Agent

In [19]:
result = agent_executor.invoke({

    "input":"Should I carry an umbrella in Islamabad?"

})

print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Islamabad'}`


Cloudy, 27°C[{'type': 'text', 'text': "The weather in Islamabad is currently cloudy with a temperature of 27°C. While it doesn't look like it's raining right now, cloudy conditions can sometimes lead to precipitation later. You might want to keep an umbrella handy just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPw2tgG4CvgHZfrOoa77oT21ofD1DQbA7uWJB2zSP/tfIEmASU+aAfVmXyl2NsIv8SBbAqN7x8f+s1kIobA88xfGJKsMFEyFyM9Mr8AV9WJPZA1eBJfO57'}}]

> Finished chain.
[{'type': 'text', 'text': "The weather in Islamabad is currently cloudy with a temperature of 27°C. While it doesn't look like it's raining right now, cloudy conditions can sometimes lead to precipitation later. You might want to keep an umbrella handy just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPw2tgG4CvgHZfrOoa77oT21ofD1DQbA7uWJB2zSP/tfIEmASU+aAfVmXyl2NsIv8SBbAqN7x8f+s1kIobA88xfGJKsMFEyFyM9Mr8AV9WJPZA1eBJfO57

# LangGraph State

LangGraph stores information inside a shared State object.

## Task

Complete the state.

💡 Hint

Today's slides showed three important fields.

One of them is

messages

In [20]:
from typing import TypedDict

class AgentState(TypedDict):
    messages: list
    city: str
    result: str

# Creating Nodes

Every node is simply a Python function.

Complete the chatbot node.

💡 Hint

The node should

1. Read the state

2. Call the LLM

3. Return the updated state

In [21]:
def chatbot_node(state):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

# Memory

Let's build a very small conversation memory.

Complete the class.

💡 Hint

Remember:

append()

returns nothing.

In [22]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)

    def get_context(self):
        return self.messages

## Challenge

Modify your memory so it only stores the latest **5** messages.

Hint:

Python list slicing may be useful.

In [23]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)
        self.messages = self.messages[-5:]

    def get_context(self):
        return self.messages

# Smart Travel Assistant

Congratulations!

You now know how to build AI Agents using LangChain.

## Your Task

Build a Travel Assistant.

Requirements

✅ Weather Tool

✅ Calculator Tool

Create ONE new tool.

Choose ONE:

- Restaurant Tool
- Movie Tool
- Hotel Tool
- Currency Converter

The assistant should answer questions like:

"I'm visiting Lahore tomorrow.

What's the weather?

Recommend a restaurant.

How much will dinner cost for 4 people if each meal costs $25?"

---

## Bonus Challenge

Can your assistant remember the user's city without asking again?

Example

User:

"I'm visiting Lahore."

Later...

"What's the weather tomorrow?"

The assistant should understand that the city is still Lahore.

In [25]:
# write code here
from langchain.tools import tool  # re-import to restore the decorator (it was overwritten earlier)

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Convert an amount from one currency to another using fixed sample rates.
    """
    rates = {
        ("USD", "PKR"): 278,
        ("PKR", "USD"): 1/278,
    }
    rate = rates.get((from_currency, to_currency))
    if rate is None:
        return "Conversion rate unavailable"
    return f"{amount} {from_currency} = {round(amount * rate, 2)} {to_currency}"

tools = [get_weather, calculator, currency_converter]
llm_with_tools = llm.bind_tools(tools)

tool_registry = {
    "get_weather": get_weather,
    "calculator": calculator,
    "currency_converter": currency_converter
}

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Simple city memory
memory = ConversationMemory()

def ask_assistant(user_input):
    memory.add({"role": "user", "content": user_input})

    # naive city extraction/carry-over
    known_cities = ["Lahore", "Karachi", "Islamabad", "Peshawar", "Quetta"]
    mentioned_city = next((c for c in known_cities if c in user_input), None)
    if mentioned_city:
        ask_assistant.last_city = mentioned_city
    elif hasattr(ask_assistant, "last_city"):
        user_input += f" (city: {ask_assistant.last_city})"

    result = agent_executor.invoke({"input": user_input})
    memory.add({"role": "assistant", "content": result["output"]})
    return result["output"]

print(ask_assistant("I'm visiting Lahore tomorrow. What's the weather?"))
print(ask_assistant("How much will dinner cost for 4 people if each meal costs $25?"))
print(ask_assistant("What's the weather tomorrow?"))  # should still know it's Lahore



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Lahore'}`


Sunny, 34°C[{'type': 'text', 'text': 'The weather in Lahore tomorrow is currently forecast to be sunny with a high of 34°C.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPwHlBmA1tsTU16wDpz4va9f6tVsEUd13zWU9PU8VVxdLv4KwhoUT5Fj4SO57kE5CppnZ+kb4CbS9xlGZuTwXirH410/wDnsfGflDRCvGNr26HC4vbWdW6'}}]

> Finished chain.
[{'type': 'text', 'text': 'The weather in Lahore tomorrow is currently forecast to be sunny with a high of 34°C.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPwHlBmA1tsTU16wDpz4va9f6tVsEUd13zWU9PU8VVxdLv4KwhoUT5Fj4SO57kE5CppnZ+kb4CbS9xlGZuTwXirH410/wDnsfGflDRCvGNr26HC4vbWdW6'}}]


> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '4 * 25'}`


100[{'type': 'text', 'text': 'Dinner for 4 people, with each meal costing $25, will cost a total of $100.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIP57E0VF9rsVsNDpzxS0jB8XahpQE+++slxAvxTS5/OTpUFLncW1R4